# Superstore Sales Analysis — Data Cleaning (Python)

**What this notebook does:** loads a retail sales dataset, checks it for problems, cleans it, adds a few useful columns, and saves a clean file for SQL analysis and dashboarding.

**How to run:** make sure `superstore.csv` is in the **same folder** as this notebook. Then run each cell top to bottom with **Shift + Enter**.

> ⚠️ **Stop at the very last (PostgreSQL) section** unless you've set up your database — everything before it runs on its own.

## Step 1 — Import the tool we need
We only need **pandas**, Python's tool for working with table data.

In [ ]:
import pandas as pd

## Step 2 — Load the data
Read the CSV file into a *DataFrame* (a table inside Python). `.head()` shows the first 5 rows.

In [ ]:
df = pd.read_csv('superstore.csv')
df.head()

## Step 3 — Explore: understand what we have
Before cleaning, look at the size, the columns, and basic stats.

In [ ]:
# how many rows and columns?
df.shape

In [ ]:
# column names, data types, and non-empty counts
df.info()

In [ ]:
# basic statistics for the number columns
df.describe()

### Check for problems
Two classic checks: missing (empty) values, and duplicate rows.

In [ ]:
# count empty values in each column
df.isnull().sum()

In [ ]:
# how many fully duplicated rows?
df.duplicated().sum()

**What we found:**
- Only `Postal Code` has empty values (11 of them).
- No duplicate rows.

So the cleaning needed here is light.

## Step 4 — Clean the data

**4a. Turn the date columns into real dates.** They load as plain text; converting them lets us pull out year/month later and sort by time.

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

**4b. Fix Postal Code.** A postal code is an *identifier*, not a number you do math on (adding two zip codes is meaningless, and numbers drop the leading zero). So we store it as **text**, and label the 11 empty ones as `'Unknown'` instead of deleting those rows (we'd lose real sales).

In [ ]:
df['Postal Code'] = df['Postal Code'].astype('Int64').astype('string')
df['Postal Code'] = df['Postal Code'].fillna('Unknown')

**4c. Remove duplicate rows.** (There are none here, but it's good practice to always do this.)

In [ ]:
df = df.drop_duplicates()

**4d. Tidy the column names.** Make everything lowercase and swap spaces/hyphens for underscores (e.g. `Order Date` becomes `order_date`). This makes the names easy to use in SQL and code.

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')
df.columns

## Step 5 — Create a few useful new columns
These make later analysis easier.

In [ ]:
# pull the year and month out of the order date (for trend-over-time charts)
df['order_year']  = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month

# how many days it took to ship each order
df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days

# profit margin = profit as a % of sales (a key business KPI)
df['profit_margin'] = (df['profit'] / df['sales'] * 100).round(2)

df.head()

## Step 6 — Quick sanity check
Confirm the new columns exist and the types look right.

In [ ]:
df.info()

## Step 7 — Save the cleaned data
This is the file you load into **Tableau** for the dashboard, and into **SQL** for the queries.

In [ ]:
df.to_csv('superstore_clean.csv', index=False)
print('Saved superstore_clean.csv with', len(df), 'rows.')

## Step 8 (Optional) — Load into PostgreSQL

> ⚠️ **Don't run this yet** unless your database is set up. Ask for help first — you'll change the connection details (username, password, database name) to match your own PostgreSQL.

The code is left here (as text) so it's ready when you are:

In [ ]:
# from sqlalchemy import create_engine
#
# # change these 4 values to match YOUR PostgreSQL setup:
# user = 'your_username'
# password = 'your_password'
# host = 'localhost'
# port = '5432'
# database = 'superstore'   # create this database first
#
# engine = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}')
# df.to_sql('superstore', engine, if_exists='replace', index=False)
# print('Loaded into PostgreSQL!')

---
**Done.** You now have a clean dataset ready for SQL and Tableau. 🎉